## Kaggle Details
this cell contains the datasets we used with the best model path.

In [ ]:
import kagglehub
kagglehub.login()
hubmap_organ_segmentation_path = kagglehub.competition_download('hubmap-organ-segmentation')
kozodoi_timm_pytorch_image_models_path = kagglehub.dataset_download('kozodoi/timm-pytorch-image-models')
igorkrashenyi_lung_hpa_dataset_path = kagglehub.dataset_download('igorkrashenyi/lung-hpa-dataset')
vladimirsydor_hubmap_2022_add_data_labels_v2_path = kagglehub.dataset_download('vladimirsydor/hubmap-2022-add-data-labels-v2')
erickgonz_erick111_path = kagglehub.dataset_download('erickgonz/erick111')

print('Data source import complete.')

: 

## Imports

In [ ]:

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)

import os, json, timeit, warnings
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.backends import cudnn

import timm

cudnn.benchmark = True



## Configurations

In [ ]:
DATA_DIR       = '/kaggle/input/competitions/hubmap-organ-segmentation'
CHECKPOINT_DIR = '/kaggle/input/datasets/erickgonz/erick111'   
SUBMISSION_FILE = 'submission.csv'

IMG_SIZE        = (768, 768)
TEST_BATCH_SIZE = 1

ORGAN_MAP  = {'kidney': 0, 'prostate': 1, 'largeintestine': 2, 'spleen': 3, 'lung': 4}
SOURCE_MAP = {'Hubmap': 0, 'HPA': 1, 'GTEx': 1}

ORGAN_THRESHOLDS = {
    'Hubmap': {'kidney': 75,  'prostate': 80, 'largeintestine': 65, 'spleen': 80, 'lung': 10},
    'HPA':    {'kidney': 127, 'prostate': 127, 'largeintestine': 127, 'spleen': 127, 'lung': 25},
    'GTEx':   {'kidney': 127, 'prostate': 127, 'largeintestine': 127, 'spleen': 127, 'lung': 25},
}

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected — attach a GPU runtime.")

print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")



## Utils and HuBMAP dataset

In [ ]:

def rle_encode_less_memory(img):
    pixels = img.T.flatten()
    pixels[0] = 0; pixels[-1] = 0
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 2
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def preprocess_inputs(x):
    x = np.asarray(x, dtype='float32')
    x /= 127.0; x -= 1.0
    return x


def _ensure_3channel(img):
    if img is None:
        return None
    if img.ndim == 2:
        return np.stack([img, img, img], axis=2)
    if img.ndim == 3 and img.shape[2] == 1:
        return np.concatenate([img, img, img], axis=2)
    if img.ndim == 3 and img.shape[2] > 3:
        return img[:, :, :3]
    return img

class HubmapTestDataset(Dataset):
    def __init__(self, df, data_dir, target_size):
        self.df = df
        self.data_dir = data_dir
        self.target_size = target_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img0 = cv2.imread(os.path.join(self.data_dir, f"{row['id']}.tiff"),
                          cv2.IMREAD_UNCHANGED)
        if img0 is None:
            raise FileNotFoundError(f"{row['id']}.tiff not found in {self.data_dir}")
        img0 = _ensure_3channel(img0)
        orig_h, orig_w = img0.shape[:2]
        img  = cv2.resize(img0, self.target_size)
        img  = preprocess_inputs(img)
        img_t = torch.from_numpy(img.transpose((2, 0, 1)).copy()).float()
        return {
            'id':          str(row['id']),
            'img':         img_t,
            'organ_name':  row['organ'],
            'source_name': row['data_source'],
            'organ_idx':   ORGAN_MAP.get(row['organ'], 5),
            'source_idx':  SOURCE_MAP.get(row['data_source'], 2),
            'pixel_size':  float(row.get('pixel_size', 0.4)),
            'orig_h':      orig_h,
            'orig_w':      orig_w,
        }



## Model Architecture

In [ ]:

class FiLM(nn.Module):
    def __init__(self, cond_dim, num_channels):
        super().__init__()
        self.fc = nn.Linear(cond_dim, num_channels * 2)

    def forward(self, x, condition):
        gb = self.fc(condition).unsqueeze(-1).unsqueeze(-1)
        g, b = torch.chunk(gb, 2, dim=1)
        return x * (1 + g) + b


class MetadataEmbedder(nn.Module):
    def __init__(self, cond_dim=128):
        super().__init__()
        self.organ_emb  = nn.Embedding(6, 32)
        self.source_emb = nn.Embedding(3, 16)
        self.scale_mlp  = nn.Sequential(nn.Linear(1, 16), nn.SiLU())
        self.fusion     = nn.Sequential(
            nn.Linear(64, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))

    def forward(self, organ_idx, source_idx, pixel_scale):
        return self.fusion(torch.cat([
            self.organ_emb(organ_idx),
            self.source_emb(source_idx),
            self.scale_mlp(pixel_scale.view(-1, 1)),
        ], dim=1))


class ConvSiluFiLM(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim, ks=3):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, ks, padding=1)
        self.silu = nn.SiLU(inplace=True)
        self.film = FiLM(cond_dim, out_ch)

    def forward(self, x, c):
        return self.film(self.silu(self.conv(x)), c)


class MultimodalPathologyUNet(nn.Module):
    def __init__(self, encoder_name='convnext_base', pretrained=False, cond_dim=128):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained,
                                         features_only=True)
        ef = [f['num_chs'] for f in self.encoder.feature_info]
        self.num_stages = len(ef)
        df = [32, 48, 64, 96, 128]
        self.metadata_embedder = MetadataEmbedder(cond_dim)
        bn = ef[-1]
        self.pred_organ  = nn.Linear(bn, 5)
        self.pred_source = nn.Linear(bn, 2)
        self.pred_scale  = nn.Linear(bn, 1)
        self.conv6   = ConvSiluFiLM(ef[-1],          df[-1], cond_dim)
        self.conv6_2 = ConvSiluFiLM(df[-1]+ef[-2],   df[-1], cond_dim)
        self.conv7   = ConvSiluFiLM(df[-1],           df[-2], cond_dim)
        self.conv7_2 = ConvSiluFiLM(df[-2]+ef[-3],   df[-2], cond_dim)
        self.conv8   = ConvSiluFiLM(df[-2],           df[-3], cond_dim)
        self.conv8_2 = ConvSiluFiLM(df[-3]+ef[-4],   df[-3], cond_dim)
        self.conv9   = ConvSiluFiLM(df[-3],           df[-4], cond_dim)
        self.conv9_2 = None if self.num_stages == 4 else \
                       ConvSiluFiLM(df[-4]+ef[-5], df[-4], cond_dim)
        self.conv10  = ConvSiluFiLM(df[-4], df[-5], cond_dim)
        self.res     = nn.Conv2d(df[-5], 1, 1)

    def forward(self, x, gt_organ=None, gt_source=None, gt_scale=None):
        feats = self.encoder(x)
        if self.num_stages == 4:
            e2, e3, e4, e5 = feats
        else:
            e1, e2, e3, e4, e5 = feats
        bn  = F.adaptive_avg_pool2d(e5, 1).view(x.shape[0], -1)
        po  = self.pred_organ(bn)
        ps  = self.pred_source(bn)
        psc = self.pred_scale(bn)
        if gt_organ is not None:
            cond = self.metadata_embedder(gt_organ, gt_source, gt_scale)
        else:
            cond = self.metadata_embedder(
                torch.argmax(po, 1), torch.argmax(ps, 1), psc)
        up = lambda t: F.interpolate(t, scale_factor=2,
                                     mode='bilinear', align_corners=False)
        d6  = self.conv6(up(e5), cond)
        d6  = self.conv6_2(torch.cat([d6, e4], 1), cond)
        d7  = self.conv7(up(d6), cond)
        d7  = self.conv7_2(torch.cat([d7, e3], 1), cond)
        d8  = self.conv8(up(d7), cond)
        d8  = self.conv8_2(torch.cat([d8, e2], 1), cond)
        d9  = self.conv9(up(d8), cond)
        if self.num_stages == 5:
            d9 = self.conv9_2(torch.cat([d9, e1], 1), cond)
        out = self.res(self.conv10(d9, cond))
        return F.interpolate(out, scale_factor=2,
                             mode='bilinear', align_corners=False)



## Inference and Submission

In [ ]:
def run_inference():
    t0 = timeit.default_timer()

    # Find checkpoint
    ckpt_path = os.path.join(CHECKPOINT_DIR, 'multimodal_unet_best.pth')
    if not os.path.exists(ckpt_path):
        # Try any .pth file in the directory
        ptfiles = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pth')]
        if not ptfiles:
            raise FileNotFoundError(f"No .pth checkpoint found in {CHECKPOINT_DIR}")
        ckpt_path = os.path.join(CHECKPOINT_DIR, ptfiles[0])
    print(f"Loading checkpoint: {ckpt_path}  ({os.path.getsize(ckpt_path)/1e6:.0f} MB)")

    ckpt  = torch.load(ckpt_path, map_location='cpu')
    model = MultimodalPathologyUNet('convnext_base', pretrained=False)
    model.load_state_dict(ckpt['state_dict'])
    model = model.eval().cuda()
    print(f"  Loaded epoch {ckpt.get('epoch', '?')}  "
          f"val_loss={ckpt.get('val_loss', '?'):.4f}")

    df      = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
    dataset = HubmapTestDataset(df, os.path.join(DATA_DIR, 'test_images'), IMG_SIZE)
    loader  = DataLoader(dataset, batch_size=TEST_BATCH_SIZE,
                         num_workers=2, shuffle=False)
    print(f"Running inference on {len(dataset)} test images with 4× TTA …\n")

    results = []
    with torch.no_grad():
        for sample in tqdm(loader, desc="Predicting"):
            img_id = sample['id'][0]
            orig_h = sample['orig_h'].item()
            orig_w = sample['orig_w'].item()
            oi = sample['organ_idx'].cuda()
            si = sample['source_idx'].cuda()
            ps = sample['pixel_size'].float().cuda()
            inp = sample['img'].numpy()
            acc = np.zeros((orig_h, orig_w), dtype='float32')

            with torch.amp.autocast('cuda'):
                for t in range(4):
                    flip = t % 2 == 1
                    rot  = t // 2
                    x = inp.copy()
                    if rot:  x = np.rot90(x, k=rot,  axes=(2, 3)).copy()
                    if flip: x = x[:, :, :, ::-1].copy()
                    out = model(torch.from_numpy(x).cuda(),
                                gt_organ=oi, gt_source=si, gt_scale=ps)
                    p = torch.sigmoid(out).float().cpu().numpy()[0, 0]
                    if flip: p = p[:, ::-1].copy()
                    if rot:  p = np.rot90(p, k=4-rot, axes=(0, 1)).copy()
                    acc += cv2.resize(p, (orig_w, orig_h))

            acc /= 4.0
            on     = sample['organ_name'][0]
            sn     = sample['source_name'][0]
            thresh = ORGAN_THRESHOLDS.get(sn, ORGAN_THRESHOLDS['HPA']).get(on, 127) / 255.0
            results.append({
                'id':  img_id,
                'rle': rle_encode_less_memory((acc > thresh).astype(np.uint8)),
            })

    sub = pd.DataFrame(results)
    sub.to_csv(SUBMISSION_FILE, index=False)
    elapsed = (timeit.default_timer() - t0) / 60
    print(f"\n✓ Submission saved to {SUBMISSION_FILE}")
    print(f"  {len(results)} predictions  |  {elapsed:.1f} min elapsed")
    print(sub.head())


run_inference()